# 第六章 · 在T4上预训练

[![在Colab中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zxh0/vibe-writing/blob/main/toyllm_book2/code/book/ch06/colab.ipynb)

这个笔记本只做几件杂事：挂载Google Drive、拉代码、把数据复制到本地盘，然后调用`book/ch06/`里的脚本。
脚本和在Mac上用`uv run`跑的是同一份。

**用之前，先把数据传到Drive上**（在Mac上第三章已经生成好了，约1.8GB）：

| Mac上的文件（`code/`下） | 传到Drive的位置 |
|---|---|
| `data/tokens/train.bin` | `我的云端硬盘/toyllm2/data/tokens/train.bin` |
| `data/tokens/val.bin` | `我的云端硬盘/toyllm2/data/tokens/val.bin` |
| `data/tokenizer/tokenizer.json` | `我的云端硬盘/toyllm2/data/tokenizer/tokenizer.json` |

**用法**：菜单“代码执行程序 → 更改运行时类型”选T4 GPU，改好第一个格子里的配置，然后“全部运行”。

* 训练记录（`log.jsonl`、`ckpt.pt`）直接写在Drive的`toyllm2/runs/<名字>/`下，会话断了也不丢。
* 免费版会断线。断了以后重新连上，再“全部运行”一次就行：`train.py`带着`--resume`，
  有checkpoint就从那一步接着训，没有就从头开始。
* 数据每次都从Drive复制到`/content/data/`。训练时要从`train.bin`里随机读，本地盘比Drive快得多。

用的是Colab自带的torch，不装书里那个`torch<2.3`（那是作者的Intel Mac的限制）。
脚本只用`torch`、`numpy`、`safetensors`、`tokenizers`，Colab都有。

In [ ]:
# ---- 配置 ----------------------------------------------------------------
REPO_URL = "https://github.com/zxh0/vibe-writing.git"
BRANCH = "main"
SUBDIR = "toyllm_book2/code"
DRIVE_DIR = "/content/drive/MyDrive/toyllm2"

RUN_BENCH = True         # 先测吞吐量：fp32、fp16、fp16加compile，B从8到64
RUN_SWEEP = False        # 学习率扫描：每个学习率训400步（约5200万词元）
SWEEP_LRS = [1e-3, 2e-3, 3e-3, 6e-3]
RUN_TRAIN = True         # 完整预训练，约5亿词元
RUN_NAME = "pretrain"
TRAIN_ARGS = []          # 额外的参数，比如 ["--lr", "2e-3", "--compile"]

In [ ]:
import os, pathlib, shutil, subprocess, sys

def sh(cmd, cwd=None):
    """跑一条shell命令，边跑边打印，失败就停。"""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

import torch
assert torch.cuda.is_available(), "没有CUDA：菜单“代码执行程序 → 更改运行时类型”选GPU"
p = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability()
print(f"GPU    {p.name}，sm{cap[0]}{cap[1]}，显存{p.total_memory / 2**30:.1f}GiB")
print(f"torch  {torch.__version__}，CUDA {torch.version.cuda}，Python {sys.version.split()[0]}")
print(f"CPU    {os.cpu_count()}个线程")
import numpy, safetensors, tokenizers   # 都是Colab自带的，缺了这里会报错

## 挂载Drive，拉代码，复制数据

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path(DRIVE_DIR)
RUNS = DRIVE / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

# 稀疏克隆：整个仓库有好几本书，这里只取第六章的代码。
WORK = pathlib.Path("/content/vibe-writing")
if not WORK.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} --filter=blob:none --no-checkout {REPO_URL} {WORK}")
    sh(f"git sparse-checkout set --no-cone /{SUBDIR}/book/ch06/", cwd=WORK)
    sh("git checkout", cwd=WORK)
else:
    sh("git pull --ff-only", cwd=WORK)
sh("git log -1 --format='%h %ad %s' --date=short", cwd=WORK)
CODE = WORK / SUBDIR

# 数据从Drive复制到本地盘。大小一样就不再复制。
DATA = pathlib.Path("/content/data")
for rel in ["tokens/train.bin", "tokens/val.bin", "tokenizer/tokenizer.json"]:
    src, dst = DRIVE / "data" / rel, DATA / rel
    assert src.exists(), f"Drive上没有{src}，先按最上面的表格传上去"
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        sh(f"cp '{src}' '{dst}'")
    print(f"{rel:<28}{dst.stat().st_size:>16,} 字节")

In [ ]:
def run(*args):
    """在code/下跑book/ch06/里的一个脚本，输出边跑边打印。"""
    cmd = [sys.executable, "-u", f"book/ch06/{args[0]}", *map(str, args[1:])]
    print("$ python", " ".join(cmd[2:]), flush=True)
    subprocess.run(cmd, cwd=CODE, check=True)

## 吞吐量

随机的词元，只测训练步本身。fp32和fp16各测一遍，再加上`torch.compile`。
结果决定正式训练用多大的batch：显存放得下、每秒词元数不再明显上升的那一档。

In [ ]:
if RUN_BENCH:
    run("bench.py", "--compile")

## 学习率扫描

第五章在Mac上用每步2,048个词元扫出来，$10^{-3}$最好。这里每步131,072个词元，batch大得多，
最合适的学习率往往也更大，要重新扫一遍。每个学习率训400步，各自一条完整的warmup加cosine。

In [ ]:
if RUN_SWEEP:
    for lr in SWEEP_LRS:
        run("train.py", "--name", f"sweep-lr{lr:g}", "--data", DATA, "--runs", RUNS,
            "--steps", 400, "--warmup", 40, "--lr", lr, "--eval-every", 100,
            "--sample-every", 400, "--ckpt-every", 100, "--resume")

## 预训练

断了重连以后，再跑一遍这个格子就会接着训。训完把模型存成第一本能读的目录：
`toyllm2/runs/<名字>/model/`。

In [ ]:
if RUN_TRAIN:
    run("train.py", "--name", RUN_NAME, "--data", DATA, "--runs", RUNS,
        "--resume", "--export", *TRAIN_ARGS)

## 看一眼曲线

画图的正式脚本留给正文，这里只是在笔记本里随手看看。

In [ ]:
import json
import matplotlib.pyplot as plt

recs = [json.loads(l) for l in (RUNS / RUN_NAME / "log.jsonl").read_text().splitlines()]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
train = [r for r in recs if "loss" in r]
val = [r for r in recs if "val_loss" in r]
axes[0].plot([r["step"] for r in train], [r["loss"] for r in train], lw=0.5, label="train")
axes[0].plot([r["step"] for r in val], [r["val_loss"] for r in val], "o-", label="val")
axes[0].set_ylabel("loss"); axes[0].legend()
axes[1].plot([r["step"] for r in train], [r["grad_norm"] for r in train], lw=0.5)
axes[1].set_ylabel("grad norm"); axes[1].set_yscale("log")
speed = [r for r in recs if "tok_per_s" in r]
axes[2].plot([r["step"] for r in speed], [r["tok_per_s"] for r in speed])
axes[2].set_ylabel("tokens/s")
for ax in axes:
    ax.set_xlabel("step")
plt.tight_layout()